In [70]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"
STAGING = PROJECT_ROOT / "data" / "staging"

for f in sorted(RAW.iterdir()):
    print(f.name)

.gitkeep
hf_memberships.parquet
hf_persons.parquet
openhalalan_winners.csv
psa_poverty.pdf
roster_legislators.pdf


## 1. hf_persons

In [71]:
persons = pd.read_parquet(RAW / "hf_persons.parquet")
print(persons.shape)
persons.head()
persons.info()

(45424, 4)
<class 'pandas.DataFrame'>
RangeIndex: 45424 entries, 0 to 45423
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           45424 non-null  str  
 1   first_name   45424 non-null  str  
 2   last_name    45424 non-null  str  
 3   name_suffix  1794 non-null   str  
dtypes: str(4)
memory usage: 3.1 MB


In [72]:
print("id unique:", persons["id"].is_unique)
print("duplicate names:", persons.duplicated(subset=["first_name", "last_name", "name_suffix"]).sum())
persons["name_suffix"].value_counts()
persons["last_name"].value_counts().head(15)

id unique: True
duplicate names: 0


last_name
Tan           165
Reyes         165
Garcia        156
Lim           140
Uy            117
De La Cruz    116
Ramos         115
Mendoza       111
Fernandez     109
Villanueva    105
Bautista      101
Flores        100
Dela Cruz      95
Gonzales       93
Santos         88
Name: count, dtype: int64

In [73]:
cruz = persons[persons["last_name"].str.contains("cruz", case=False)]
cruz["last_name"].value_counts()

last_name
De La Cruz       116
Dela Cruz         95
Cruz              85
Vera Cruz          3
Cruzado            3
Cruzat             2
Cruz-Am            2
Sta Cruz           1
Salva Cruz         1
Veracruz           1
Inza Cruz          1
Cruzada            1
Cruza              1
Dela Cruz Jr       1
Escanlar-Cruz      1
Name: count, dtype: int64

In [74]:
print("extra spaces:", (persons["last_name"] != persons["last_name"].str.strip()).sum())
print("all caps:", persons["last_name"].str.isupper().sum())

extra spaces: 0
all caps: 0


In [75]:
suffix_in_last = persons["last_name"].str.contains(r"\s(?:Jr|Sr|II|III|IV)\.?$", regex=True)
print("suffix inside last_name:", suffix_in_last.sum())
persons.loc[suffix_in_last, ["last_name", "name_suffix"]].head(10)


suffix inside last_name: 6


,last_name,name_suffix
9860,Gabuya Jr,NaN
22456,Pineda Jr,NaN
32000,Dela Cruz Jr,NaN
32062,De Manuel Sr,NaN
35868,Lopez Jr,NaN
43472,Rapanut Sr,NaN


In [76]:
persons["name_suffix"].value_counts()

name_suffix
Jr.       1287
Sr.        224
III        135
II          80
V           25
IV          21
I           14
VI           3
VII          1
Jr. IV       1
XVI          1
XIV          1
Jr. II       1
Name: count, dtype: int64

**What it is:** One row per politician. 45,424 rows, 4 columns (`id`, `first_name`, `last_name`, `name_suffix`).

**Quality**
- `id` is unique, so it works as the primary key.
- `first_name` and `last_name` have no missing values, no leading/trailing spaces, and no all-caps entries.
- `name_suffix` is filled for 1,794 rows (about 4%). This is expected, since most people have no suffix.

**Issues found**
- **Surname variants:** the same name is written differently, e.g. "De La Cruz" (116) vs "Dela Cruz" (95), "Vera Cruz" vs "Veracruz", and abbreviations such as "Sta Cruz".
- **Hyphenated surnames** (e.g. "Escanlar-Cruz") may combine two family names.
- **Suffixes inside `last_name`:** 6 rows (e.g. "Lopez Jr", "Rapanut Sr") have the suffix in the last name while `name_suffix` is empty.
- **Odd suffix values:** formatting is consistent ("Jr.", "Sr.", Roman numerals), but "XVI" and "XIV" (1 each) are implausible and "I" (14) is unusual.
- **Zero duplicate full names** across 45,424 people is unusually clean. The source may have merged different people who share a name (to verify using memberships).
- **Common surnames** (Tan 165, Reyes 165, Garcia 156, Lim 140, Uy 117) are shared by unrelated families, so a surname alone is a weak kinship signal.

**Not in this table:** no middle name and no location. Location must come from `hf_memberships`.

## 2. hf_memberships

In [77]:
memberships = pd.read_parquet(RAW / "hf_memberships.parquet")
print(memberships.shape)
memberships.info()

(86234, 8)
<class 'pandas.DataFrame'>
RangeIndex: 86234 entries, 0 to 86233
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         86234 non-null  str  
 1   person_id  86234 non-null  str  
 2   party      86234 non-null  str  
 3   region     86234 non-null  str  
 4   province   86234 non-null  str  
 5   locality   58594 non-null  str  
 6   position   86234 non-null  str  
 7   year       86234 non-null  int32
dtypes: int32(1), str(7)
memory usage: 13.0 MB


In [78]:
orphans = ~memberships["person_id"].isin(persons["id"])
print("memberships with no matching person:", orphans.sum())

memberships with no matching person: 0


In [79]:
memberships["year"].value_counts().sort_index()

year
2004    17372
2007    16610
2010    17507
2013    17164
2016    17581
Name: count, dtype: int64

In [80]:
memberships.loc[memberships["locality"].isna(), "position"].value_counts()

position
COUNCILOR                           17678
PROVINCIAL BOARD MEMBER              3745
MAYOR                                2174
VICE MAYOR                           2170
MEMBER, HOUSE OF REPRESENTATIVES     1082
GOVERNOR                              397
VICE GOVERNOR                         394
Name: count, dtype: int64

In [81]:
pd.crosstab(memberships["position"], memberships["locality"].isna())

locality,False,True
position,,
COUNCILOR,47114,17678
GOVERNOR,0,397
MAYOR,5727,2174
"MEMBER, HOUSE OF REPRESENTATIVES",39,1082
PROVINCIAL BOARD MEMBER,2,3745
VICE GOVERNOR,0,394
VICE MAYOR,5712,2170


In [82]:
town_level = memberships[memberships["position"].isin(["MAYOR", "VICE MAYOR", "COUNCILOR"])]
pd.crosstab(town_level["year"], town_level["locality"].isna())

locality,False,True
year,,
2004,16259,0
2007,15483,0
2010,16380,0
2013,201,15842
2016,10230,6180


In [83]:
town_level.loc[town_level["locality"].isna(), "province"].value_counts().head(10)

province
CEBU             1084
BOHOL             931
CAMARINES SUR     735
BATANGAS          679
CAGAYAN           588
ABRA              540
BULACAN           495
PANGASINAN        478
CAVITE            470
ILOILO            446
Name: count, dtype: int64

In [84]:
t2016 = town_level[town_level["year"] == 2016]
share_missing = t2016["locality"].isna().groupby(t2016["province"]).mean()
share_missing[share_missing > 0].sort_index()

province
ABRA                                   1.000000
AGUSAN DEL NORTE                       1.000000
AGUSAN DEL SUR                         1.000000
AKLAN                                  1.000000
ALBAY                                  1.000000
ANTIQUE                                1.000000
APAYAO                                 1.000000
AURORA                                 1.000000
BASILAN                                1.000000
BATAAN                                 1.000000
BATANES                                1.000000
BATANGAS                               1.000000
BENGUET                                1.000000
BILIRAN                                1.000000
BOHOL                                  1.000000
BUKIDNON                               1.000000
BULACAN                                1.000000
CAGAYAN                                1.000000
CAMARINES NORTE                        1.000000
CAMARINES SUR                          1.000000
CAMIGUIN                       

In [85]:
full = share_missing[share_missing == 1].sort_index()
print(len(full), "provinces fully missing, from", full.index[0], "to", full.index[-1])
share_missing[(share_missing > 0) & (share_missing < 1)]

36 provinces fully missing, from ABRA to NCR, CITY OF MANILA, FIRST DISTRICT


province
NCR, FOURTH DISTRICT    0.131579
NCR, SECOND DISTRICT    0.530612
SULTAN KUDARAT          0.008130
Name: locality, dtype: float64

In [86]:
all_2016 = sorted(t2016["province"].unique())
cutoff = full.index[-1]
before_cutoff = [p for p in all_2016 if p <= cutoff]

print(len(before_cutoff), "provinces from A up to the cutoff")
[p for p in before_cutoff if p not in full.index]

52 provinces from A up to the cutoff


['ILOCOS NORTE',
 'ILOCOS SUR',
 'ILOILO',
 'ISABELA',
 'KALINGA',
 'LA UNION',
 'LAGUNA',
 'LANAO DEL NORTE',
 'LANAO DEL SUR',
 'LEYTE',
 'MAGUINDANAO',
 'MARINDUQUE',
 'MASBATE',
 'MISAMIS OCCIDENTAL',
 'MISAMIS ORIENTAL',
 'MOUNTAIN PROVINCE']

In [87]:
full = share_missing[share_missing == 1].sort_index()
print(len(full), "provinces fully missing, from", full.index[0], "to", full.index[-1])
share_missing[(share_missing > 0) & (share_missing < 1)]

36 provinces fully missing, from ABRA to NCR, CITY OF MANILA, FIRST DISTRICT


province
NCR, FOURTH DISTRICT    0.131579
NCR, SECOND DISTRICT    0.530612
SULTAN KUDARAT          0.008130
Name: locality, dtype: float64

In [88]:
print("Last province in the A block:", full.index[-2])

Last province in the A block: IFUGAO


In [89]:
known = memberships.dropna(subset=["locality"])
missing = town_level[town_level["locality"].isna()]

has_history = missing["person_id"].isin(known["person_id"])
print("missing rows:", len(missing))
print("person has a known town in another year:", has_history.sum())

missing rows: 22022
person has a known town in another year: 13924


In [90]:
towns_per_person = known.groupby(["person_id", "province"])["locality"].nunique()

check = missing.merge(
    towns_per_person.rename("n_towns"),
    left_on=["person_id", "province"],
    right_index=True,
    how="left",
)
check["n_towns"].value_counts(dropna=False)

n_towns
1.0    13748
NaN     8194
2.0       79
3.0        1
Name: count, dtype: int64

### Summary: hf_memberships

**What it is:** One row per person, per position, per election. 86,234 rows, 8 columns (`id`, `person_id`, `party`, `region`, `province`, `locality`, `position`, `year`).

**Quality**
- Every `person_id` matches an `id` in `hf_persons` (0 orphans), so the two tables join cleanly.
- 5 elections, evenly covered: 2004 (17,372), 2007 (16,610), 2010 (17,507), 2013 (17,164), 2016 (17,581). No data after 2016.
- About 1,580 mayor rows per election, consistent with the roughly 1,600 Philippine cities and municipalities.

**Missing `locality` (27,640 rows, 32%)**
- **Expected blanks (5,618):** Provincial Board Member (3,745), House Representative (1,082), Governor (397), and Vice Governor (394) cover a province or district, not one town.
- **Real gaps (22,022):** Councilor (17,678), Mayor (2,174), and Vice Mayor (2,170), each about 27% missing.
- **The gaps are concentrated in two years:**

| Year | Town-level rows missing locality |
|---|---|
| 2004, 2007, 2010 | 0% |
| 2013 | 15,842 of 16,043 (about 99%) |
| 2016 | 6,180 of 16,410 (about 38%) |

- **2013:** locality is missing almost everywhere, so the source essentially did not record towns for this election.
- **2016:** 36 provinces are fully missing, in two separate patterns:
  - **An alphabetical block of 35 provinces**, from ABRA to IFUGAO, all 100% missing. The 16 provinces from ILOCOS NORTE to MOUNTAIN PROVINCE are complete, as are the provinces after NCR. This suggests the first part of an alphabetical collection run failed.
  - **NCR districts:** NCR, City of Manila, First District (100% missing), NCR Second District (53%), NCR Fourth District (13%). NCR Third District is complete. This appears to be a separate issue from the alphabetical block.
  - Sultan Kudarat is also slightly affected (0.8%).

**Backfill potential from each person's own history** (same person, same province, town known in another year):

| Group | Rows | Share |
|---|---|---|
| Exactly one known town | 13,748 | 62.4% |
| Two or three known towns (ambiguous) | 80 | 0.4% |
| No known town in that province | 8,194 | 37.2% |

96 of the 8,194 have a known town in a *different* province (they moved, or the province name differs between years). Rows that cannot be filled from history will need another source (OpenHalalan, section 3).

**Other observations**
- 39 House Representatives and 2 Provincial Board Members have a locality (to check).
- `position` values are written in ALL CAPS.
- NCR uses district-style province names, e.g. "NCR, CITY OF MANILA, FIRST DISTRICT".

## 3. openhalalan_winners

In [91]:
winners = pd.read_csv(RAW / "openhalalan_winners.csv")
print(winners.shape)
winners.info()

(157333, 14)
<class 'pandas.DataFrame'>
RangeIndex: 157333 entries, 0 to 157332
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   Last Name           157326 non-null  str  
 1   First Name          157251 non-null  str  
 2   Middle Name         139887 non-null  str  
 3   Middle Name Source  129693 non-null  str  
 4   Title               1140 non-null    str  
 5   Full Name           157333 non-null  str  
 6   Position            157333 non-null  str  
 7   Party               157203 non-null  str  
 8   Year                157333 non-null  int64
 9   Province            157281 non-null  str  
 10  City                143744 non-null  str  
 11  Region              157281 non-null  str  
 12  Sex                 148681 non-null  str  
 13  Sex Source          148681 non-null  str  
dtypes: int64(1), str(13)
memory usage: 31.4 MB


In [92]:
winners["Year"].value_counts().sort_index()

Year
2001    17490
2004    17368
2007    16608
2010    17502
2013    17161
2016    17774
2019    17793
2022    17818
2025    17819
Name: count, dtype: int64

In [93]:
pd.crosstab(winners["Year"], winners["City"].isna())

City,False,True
Year,,
2001,16394,1096
2004,15522,1846
2007,14912,1696
2010,15452,2050
2013,14905,2256
2016,16625,1149
2019,16630,1163
2022,16644,1174
2025,16660,1159


In [94]:
w2013 = winners[winners["Year"] == 2013]
w2013.loc[w2013["City"].isna(), "Position"].value_counts()

Position
COUNCILOR                           905
PROVINCIAL BOARD MEMBER             725
MEMBER, HOUSE OF REPRESENTATIVES    266
MAYOR                               124
VICE MAYOR                           85
VICE GOVERNOR                        78
GOVERNOR                             73
Name: count, dtype: int64

In [95]:
winners["Position"].value_counts()

Position
COUNCILOR                           118203
VICE MAYOR                           14419
MAYOR                                14398
PROVINCIAL BOARD MEMBER               6737
MEMBER, HOUSE OF REPRESENTATIVES      2108
VICE GOVERNOR                          714
GOVERNOR                               702
SENATOR                                 48
PRESIDENT                                2
VICE PRESIDENT                           2
Name: count, dtype: int64

In [96]:
no_province = winners[winners["Province"].isna()]
no_province["Position"].value_counts()

Position
SENATOR           48
PRESIDENT          2
VICE PRESIDENT     2
Name: count, dtype: int64

In [97]:
winners["Middle Name Source"].value_counts()

Middle Name Source
original       63033
comelec_lec    44986
dilg           13954
self-prior      7408
v8.5             312
Name: count, dtype: int64

In [98]:
hf_prov = set(memberships["province"].unique())
oh_prov = set(winners["Province"].dropna().unique())

print("HF provinces:", len(hf_prov), "| OpenHalalan provinces:", len(oh_prov))
print("Only in HF:", sorted(hf_prov - oh_prov))
print("Only in OpenHalalan:", sorted(oh_prov - hf_prov))

HF provinces: 85 | OpenHalalan provinces: 88
Only in HF: ['COMPOSTELA VALLEY', 'NCR, CITY OF MANILA, FIRST DISTRICT', 'NCR, FOURTH DISTRICT', 'NCR, SECOND DISTRICT', 'NCR, THIRD DISTRICT', 'TAWI-TAWI']
Only in OpenHalalan: ['DAVAO DE ORO', 'MAGUINDANAO DEL NORTE', 'MAGUINDANAO DEL SUR', 'NCR FIRST DISTRICT', 'NCR FOURTH DISTRICT', 'NCR SECOND DISTRICT', 'NCR THIRD DISTRICT', 'SPECIAL GEOGRAPHIC AREA', 'TAWI TAWI']


In [99]:
check_prov = ["DAVAO DE ORO", "MAGUINDANAO", "MAGUINDANAO DEL NORTE",
              "MAGUINDANAO DEL SUR", "SPECIAL GEOGRAPHIC AREA"]
winners[winners["Province"].isin(check_prov)].groupby("Province")["Year"].unique()

Province
DAVAO DE ORO               [2001, 2004, 2007, 2010, 2013, 2016, 2019, 202...
MAGUINDANAO                 [2001, 2004, 2007, 2010, 2013, 2016, 2019, 2022]
MAGUINDANAO DEL NORTE                                                 [2025]
MAGUINDANAO DEL SUR                                                   [2025]
SPECIAL GEOGRAPHIC AREA                                               [2025]
Name: Year, dtype: object

In [100]:
hf_towns = set(memberships["locality"].dropna().unique())
oh_towns = set(winners["City"].dropna().unique())

only_hf = sorted(hf_towns - oh_towns)
only_oh = sorted(oh_towns - hf_towns)

print("HF towns:", len(hf_towns), "| OpenHalalan towns:", len(oh_towns))
print("Only in HF:", len(only_hf), "| Only in OpenHalalan:", len(only_oh))
print("Sample only in HF:", only_hf[:20])
print("Sample only in OpenHalalan:", only_oh[:20])

HF towns: 1463 | OpenHalalan towns: 1517
Only in HF: 254 | Only in OpenHalalan: 308
Sample only in HF: ['AL-BARKA', 'ALAMINOS CITY', 'ALFONSO LISTA (POTIA)', 'AMLAN (AYUQUITAN)', 'ANGELES CITY', 'ANINI-Y', 'ASUNCION (SAUG)', 'BACOLOD CITY', 'BACOLOD-KALAWI (BACOLOD GRANDE)', 'BACUNGAN (LEON T. POSTIGO)', 'BACUNGAN (Leon T. Postigo)', 'BAGO CITY', 'BAGUIO CITY', 'BAIS CITY', 'BALAGTAS (BIGAA)', 'BALANGA CITY', 'BALINDONG (WATU)', 'BANNA (ESPIRITU)', 'BASILISA (RIZAL)', 'BATAC CITY']
Sample only in OpenHalalan: ['AL BARKA', 'ALFONSO LISTA', 'ALFONSO LISTA POTIA', 'AMAI MANABILANG', 'AMAI MANABILANG BUMBARAN', 'AMLAN', 'AMLAN AYUQUITAN', 'ANGELES', 'ANINI Y', 'ANTIPOLO', 'ASUNCION', 'ASUNCION SAUG', 'BACOLOD KALAWI', 'BACOLOD KALAWI BACOLOD GRANDE', 'BACUNGAN', 'BACUNGAN LEON T POSTIGO', 'BAGO', 'BAGUIO', 'BAIS', 'BALAGTAS']


In [101]:
def norm_town(name):
    name = name.upper()
    name = re.sub(r"\(.*?\)", "", name)
    name = name.replace("Ñ", "N")                                  # new: Ñ → N
    name = name.replace("-", " ").replace(".", "").replace("'", "")  # new: also removes apostrophes
    name = re.sub(r"\s+", " ", name).strip()                       # moved: clean spaces BEFORE the CITY rules
    name = re.sub(r"^CITY OF ", "", name)
    name = re.sub(r" CITY$", "", name)
    return name

In [102]:
hf_norm = {norm_town(t) for t in hf_towns}
oh_norm = {norm_town(t) for t in oh_towns}

still_hf = sorted(hf_norm - oh_norm)
still_oh = sorted(oh_norm - hf_norm)

print("Before cleaning:", len(only_hf), "|", len(only_oh))
print("After cleaning:", len(still_hf), "|", len(still_oh))
print("Still only in HF:", still_hf[:20])
print("Still only in OpenHalalan:", still_oh[:20])

Before cleaning: 254 | 308
After cleaning: 5 | 120
Still only in HF: ['MAGADALENA', 'PATEROS TAGUIG', 'PRES CARLOS P GARCIA', 'SAM NARCISO', 'TAGUIG PATEROS']
Still only in OpenHalalan: ['ALFONSO LISTA POTIA', 'AMAI MANABILANG', 'AMAI MANABILANG BUMBARAN', 'AMLAN AYUQUITAN', 'ASUNCION SAUG', 'BACOLOD KALAWI BACOLOD GRANDE', 'BACUNGAN LEON T POSTIGO', 'BALAGTAS BIGAA', 'BALINDONG WATU', 'BALIWAG', 'BANAY BANAY', 'BASILISA RIZAL', 'BAYAWAN CITY TULONG', 'BINDOY PAYABON', 'BROOKE S POINT', 'BULALACAO SAN PEDRO', 'CALANASAN BAYAG', 'CORDOVA', 'D V CHIONGBIAN D M MARCOS', 'DARAGA LOCSIN']


In [103]:
winners["town_norm"] = winners["City"].map(norm_town, na_action="ignore")

overlap = winners[winners["Year"].between(2004, 2016)]
affected = overlap[overlap["town_norm"].isin(still_oh)]

print("Overlap-year rows with a leftover name:", len(affected), "of", len(overlap))
print("Distinct leftover names in overlap years:", affected["town_norm"].nunique())

Overlap-year rows with a leftover name: 3875 of 86413
Distinct leftover names in overlap years: 96


### Summary: openhalalan_winners

**What it is:** Election winners. 157,333 rows, 14 columns, covering 9 elections from 2001 to 2025 (about 17,000 rows each). Column names use Title Case with spaces (e.g. `Last Name`), unlike the snake_case names in HF.

**Strengths**
- **Town (`City`) is 91% complete**, including the years HF is missing: 2013 (14,905 of 17,161 rows, 87%) and 2016 (16,625 of 17,774 rows, 94%).
- **Has `Middle Name` (89% complete)**, which HF lacks. It is a useful kinship clue, since the middle name is usually the mother's maiden surname.

**Middle name sources**

| Source | Rows |
|---|---|
| original | 63,033 |
| comelec_lec | 44,986 |
| dilg | 13,954 |
| self-prior | 7,408 |
| v8.5 | 312 |

10,194 middle names have no recorded source. The meaning of `self-prior` and `v8.5` still needs to be confirmed from OpenHalalan's documentation.

**Gaps**
- In 2013, 1,114 town-level rows (905 Councilors, 124 Mayors, 85 Vice Mayors) have no town in this source either.
- 7 missing last names and 82 missing first names.
- The 52 rows with no Province/Region are national positions (48 Senators, 2 Presidents, 2 Vice Presidents). National coverage looks incomplete, so these rows are not useful for town-level analysis.
- `Sex`, `Sex Source`, and `Title` are not yet profiled.

**Matching with HF (2004–2016)**
- **No person ID**, so rows must be matched on name, province, year, and position.
- **Row counts are nearly identical** in shared years (within 5 rows, except 2016: 17,774 vs 17,581). The two sources likely share an origin, so their agreement is not independent confirmation.
- **Position names match HF exactly.** OpenHalalan also has SENATOR, PRESIDENT, VICE PRESIDENT.
- **Provinces:** 6 mappings needed: 4 NCR districts (comma and naming differences), TAWI-TAWI vs TAWI TAWI, and COMPOSTELA VALLEY (HF) vs DAVAO DE ORO (OpenHalalan, used for all years).
- **Towns:** mismatches dropped from 254 / 308 to 5 / 120 after the `norm_town` cleaning function (uppercase, remove parentheses, Ñ → N, remove hyphens, periods, and apostrophes, remove "CITY OF" / " CITY"). In shared years, 3,875 of 86,413 rows (4.5%) still use a leftover name across 96 distinct names. Together with HF's 5, about 100 manual mappings are needed.

**Open questions for the team**
- Use OpenHalalan's extra years (2001, 2019–2025), or limit the analysis to 2004–2016?
- How to handle provinces created after 2016: Maguindanao del Norte and del Sur (appear only in 2025; MAGUINDANAO runs through 2022) and Special Geographic Area (2025)?

## 4. psa_poverty (staging)

In [104]:
poverty = pd.read_csv(STAGING / "psa_poverty.csv")
print(poverty.shape)
poverty.info()

(320, 13)
<class 'pandas.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   area               320 non-null    str    
 1   area_key           320 non-null    str    
 2   level              320 non-null    str    
 3   region             317 non-null    str    
 4   year               320 non-null    int64  
 5   poverty_incidence  320 non-null    float64
 6   poverty_threshold  320 non-null    float64
 7   cv                 320 non-null    float64
 8   std_error          320 non-null    float64
 9   ci_lower           320 non-null    float64
 10  ci_upper           320 non-null    float64
 11  flags              291 non-null    str    
 12  page               320 non-null    int64  
dtypes: float64(6), int64(2), str(5)
memory usage: 54.6 KB


In [105]:
poverty.head(10)

,area,area_key,level,region,year,poverty_incidence,poverty_threshold,cv,std_error,ci_lower,ci_upper,flags,page
0,PHILIPPINES,PHILIPPINES,national,NaN,2018,12.1,25814.0,1.4,0.2,11.8,12.5,"r1, 1/, 2/, 3/",18
1,PHILIPPINES,PHILIPPINES,national,NaN,2021,13.2,28794.0,1.4,0.2,12.8,13.5,"r1, 1/, 2/, 3/",18
2,PHILIPPINES,PHILIPPINES,national,NaN,2023,10.9,33296.0,1.5,0.2,10.6,11.2,"r1, 1/, 2/, 3/",18
3,National Capital Region (NCR),NATIONAL CAPITAL REGION,region,National Capital Region (NCR),2018,1.4,28682.0,10.4,0.1,1.1,1.7,"1/, 2/, a/",18
4,National Capital Region (NCR),NATIONAL CAPITAL REGION,region,National Capital Region (NCR),2021,2.2,32978.0,6.7,0.1,1.9,2.5,"1/, 2/, a/",18
5,National Capital Region (NCR),NATIONAL CAPITAL REGION,region,National Capital Region (NCR),2023,1.1,37711.0,9.9,0.1,0.9,1.4,"1/, 2/, a/",18
6,1st District,1ST DISTRICT,district,National Capital Region (NCR),2018,1.9,28682.0,30.8,0.6,0.8,3.1,"a/, b/, c/",18
7,1st District,1ST DISTRICT,district,National Capital Region (NCR),2021,1.1,32978.0,32.9,0.4,0.4,1.8,"a/, b/, c/",18
8,1st District,1ST DISTRICT,district,National Capital Region (NCR),2023,1.6,37711.0,29.6,0.5,0.7,2.5,"a/, b/, c/",18
9,2nd District,2ND DISTRICT,district,National Capital Region (NCR),2018,1.4,28682.0,20.1,0.3,0.9,2.0,"2/, a/, b/, c/",18


In [106]:
poverty["year"].value_counts().sort_index()

year
2018    106
2021    106
2023    108
Name: count, dtype: int64

In [107]:
poverty["level"].value_counts()

level
province    251
region       54
district     12
national      3
Name: count, dtype: int64

In [108]:
names_by_year = poverty.groupby("year")["area_key"].apply(set)

print("Only in 2023:", sorted(names_by_year[2023] - names_by_year[2021]))
print("In 2021 but not 2023:", sorted(names_by_year[2021] - names_by_year[2023]))

Only in 2023: ['MAGUINDANAO DEL NORTE', 'MAGUINDANAO DEL SUR']
In 2021 but not 2023: []


In [109]:
poverty[poverty["area_key"].str.contains("MAGUINDANAO")][
    ["area", "level", "year", "poverty_incidence", "flags", "page"]
]

,area,level,year,poverty_incidence,flags,page
306,Maguindanao,province,2018,40.6,"1/, 3/",21
307,Maguindanao,province,2021,29.8,"1/, 3/",21
308,Maguindanao,province,2023,30.4,"1/, 3/",21
309,Maguindanao del Norte,province,2023,28.4,NaN,21
310,Maguindanao del Sur,province,2023,32.1,NaN,21


In [110]:
psa_prov = set(poverty.loc[poverty["level"] == "province", "area_key"])

print("PSA provinces:", len(psa_prov))
print("Only in PSA:", sorted(psa_prov - oh_prov))
print("Only in OpenHalalan:", sorted(oh_prov - psa_prov))

PSA provinces: 85
Only in PSA: ['COTABATO CITY', 'ISABELA CITY']
Only in OpenHalalan: ['NCR FIRST DISTRICT', 'NCR FOURTH DISTRICT', 'NCR SECOND DISTRICT', 'NCR THIRD DISTRICT', 'SPECIAL GEOGRAPHIC AREA']


In [111]:
winners.loc[winners["town_norm"].isin(["COTABATO", "ISABELA"]), ["Province", "town_norm"]].drop_duplicates()

,Province,town_norm
1317,BASILAN,ISABELA
9571,MAGUINDANAO,COTABATO
10998,NEGROS OCCIDENTAL,ISABELA
149211,MAGUINDANAO DEL NORTE,COTABATO


In [112]:
dupes = poverty.duplicated(subset=["area_key", "level", "year"], keep=False)
print("duplicate area-year rows:", dupes.sum())
poverty.loc[dupes, ["area", "area_key", "level", "year", "poverty_incidence", "page"]]

duplicate area-year rows: 0


,area,area_key,level,year,poverty_incidence,page


### Summary: psa_poverty (staging)

**What it is:** PSA poverty statistics, parsed from `psa_poverty.pdf` into a CSV (staging). 320 rows, 13 columns, in long format: one row per area, per level, per year.

**Columns:** `area`, `area_key` (uppercased name for matching), `level`, `region`, `year`, `poverty_incidence`, `poverty_threshold`, `cv`, `std_error`, `ci_lower`, `ci_upper`, `flags` (PDF footnote marks), `page` (source PDF page).

**Quality**
- All number columns were parsed as numbers, not text.
- No duplicate rows: each area appears once per level per year.
- `page` records the source PDF page for every row, so values can be traced back to the original document.
- `region` is missing only for the 3 national rows (expected). `flags` is empty in 29 rows (no footnotes).

**Coverage**
- **Years:** 2018 (106 rows), 2021 (106), 2023 (108).
- **Levels:** province (251), region (54), district (12, the NCR districts), national (3).

**Key limitations**
- **No overlap with HF (2004–2016).** The nearest elections are OpenHalalan's 2019, 2022, and 2025.
- **The smallest level is province.** There is no town-level poverty data, except Isabela City and Cotabato City, which PSA reports separately as if they were provinces.
- **Precision varies.** Some estimates have a high `cv` (e.g. about 30 for NCR 1st District), so they are less reliable.
- The national values (12.1, 13.2, 10.9) appear to match PSA's poverty incidence among **families**, not individuals (to confirm in the PDF).

**Maguindanao in 2023**
- 2023 has three rows: Maguindanao (30.4), Maguindanao del Norte (28.4), and Maguindanao del Sur (32.1).
- The Maguindanao value lies between the two new provinces, so it is likely a combined figure kept for comparison with 2018 and 2021. Using all three rows together would count Maguindanao twice.

**Matching with OpenHalalan provinces**
- PSA's 85 province names match OpenHalalan's exactly, except:
  - **Only in PSA:** COTABATO CITY and ISABELA CITY. In OpenHalalan these are towns: Isabela in BASILAN, and Cotabato in MAGUINDANAO (through 2022) and MAGUINDANAO DEL NORTE (2025).
  - **Only in OpenHalalan:** the 4 NCR districts (PSA labels them `district`, with names like "1ST DISTRICT") and SPECIAL GEOGRAPHIC AREA (no PSA figure).
- A town named ISABELA also exists in NEGROS OCCIDENTAL, so towns must always be matched by province **and** town together.

**To confirm in the PDF**
- What each flag means (footnotes on the PDF pages, e.g. pages 18 and 21).
- Whether the Basilan and Maguindanao figures exclude Isabela City and Cotabato City.
- Whether `poverty_incidence` refers to families or individuals.

**Open question for the team**
- Option A: link poverty to OpenHalalan's 2019, 2022, and 2025 elections (nearest poverty year for each).
- Option B: use poverty only as background context, without year-by-year linking.

## 5. roster_legislators (staging)

In [113]:
roster = pd.read_csv(STAGING / "roster_legislators.csv")
print(roster.shape)
roster.info()

(5118, 15)
<class 'pandas.DataFrame'>
RangeIndex: 5118 entries, 0 to 5117
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             5118 non-null   str    
 1   surname          5118 non-null   str    
 2   region_province  5054 non-null   str    
 3   district         4448 non-null   str    
 4   period_clean     5117 non-null   str    
 5   congress_number  3217 non-null   float64
 6   start_year       5115 non-null   float64
 7   end_year         5115 non-null   float64
 8   years_raw        5115 non-null   str    
 9   years_corrected  5118 non-null   bool   
 10  years_suspect    5118 non-null   bool   
 11  notes            387 non-null    str    
 12  is_party_list    5118 non-null   bool   
 13  is_sectoral      5118 non-null   bool   
 14  page             5118 non-null   int64  
dtypes: bool(4), float64(3), int64(1), str(7)
memory usage: 831.9 KB


In [114]:
roster.head(10)

,name,surname,region_province,district,period_clean,congress_number,start_year,end_year,years_raw,years_corrected,years_suspect,notes,is_party_list,is_sectoral,page
0,"ABAD, AURORA B.",ABAD,Batanes,Lone,6th Congress,6.0,1966.0,1969.0,1966-1969,False,False,NaN,False,False,1
1,"ABAD, FLORENCIO B.",ABAD,Batanes,Lone,8th Congress,8.0,1987.0,1992.0,1987-1992,False,False,NaN,False,False,1
2,"ABAD, FLORENCIO B.",ABAD,Batanes,Lone,10th Congress,10.0,1995.0,1998.0,1995-1998,False,False,NaN,False,False,1
3,"ABAD, FLORENCIO B.",ABAD,Batanes,Lone,11th Congress,11.0,1998.0,2001.0,1998-2001,False,False,NaN,False,False,1
4,"ABAD, FLORENCIO B.",ABAD,Batanes,Lone,12th Congress,12.0,2001.0,2004.0,2001-2004,False,False,NaN,False,False,1
5,"ABAD, HENEDINA R.",ABAD,Batanes,Lone,13th Congress,13.0,2004.0,2007.0,2004-2007,False,False,NaN,False,False,1
6,"ABAD, HENEDINA R.",ABAD,Batanes,Lone,15th Congress,15.0,2010.0,2013.0,2010-2013,False,False,NaN,False,False,1
7,"ABAD, HENEDINA R.",ABAD,Batanes,Lone,16th Congress,16.0,2013.0,2016.0,2013-2016,False,False,NaN,False,False,1
8,"ABAD, HENEDINA R.",ABAD,Batanes,Lone,17th Congress,17.0,2016.0,2017.0,2016-2017*,False,False,"*Died on October 8, 2017 during her term of of...",False,False,1
9,"ABAD, JORGE A.",ABAD,Batanes,Lone,2nd Congress,2.0,1949.0,1953.0,1949-1953,False,False,NaN,False,False,1


In [115]:
roster.loc[roster["congress_number"].isna(), "period_clean"].value_counts()

period_clean
Regular Batasang Pambansa               197
Interim Batasang Pambansa               188
Batasang Bayan                          140
National Assembly                       111
1st Commonwealth National Assembly       99
9th Philippine Legislature               99
2nd Commonwealth National Assembly       98
8th Philippine Legislature               96
10th Philippine Legislature              95
6th Philippine Legislature               93
1st Congress of Commonwealth             93
7th Philippine Legislature               92
4th Philippine Legislature               89
5th Philippine Legislature               89
3rd Philippine Legislature               88
2nd Philippine Legislature               85
1st Philippine Legislature               83
First Congress                           24
5th Philipine Legislature                 5
1st Congress Commonwealth                 3
Seventh Congress                          3
13th Congresss                            3
12th congress      

In [116]:
print("years_corrected:", roster["years_corrected"].sum())
print("years_suspect:", roster["years_suspect"].sum())
print("party list:", roster["is_party_list"].sum())
print("sectoral:", roster["is_sectoral"].sum())

years_corrected: 5
years_suspect: 5
party list: 273
sectoral: 80


In [117]:
no_prov = roster[roster["region_province"].isna()]
print("rows with no province:", len(no_prov))
print("of those, party list or sectoral:", (no_prov["is_party_list"] | no_prov["is_sectoral"]).sum())

rows with no province: 64
of those, party list or sectoral: 0


In [118]:
no_prov[["name", "district", "period_clean", "start_year", "notes", "page"]].head(15)

,name,district,period_clean,start_year,notes,page
15,"ABAD SANTOS, VICENTE B.",NaN,Batasang Bayan,1976.0,*Member of the Cabinet (Secretary of Justice).,2
190,"ALBA, MANUEL S.",NaN,Regular Batasang Pambansa,1984.0,*Member of the Cabinet (Office of Budget and M...,9
212,"ALBERTO, JOSE M.",NaN,Batasang Bayan,1976.0,*Regional Representative-at-Large.,10
226,"ALDABA-LIM, ESTEFANIA",NaN,Batasang Bayan,1976.0,*Member of The Cabinet (Secretary of Social Se...,11
239,"ALDEGUER, JOSE M.",NaN,Batasang Bayan,1976.0,*Regional Representative-at-Large.,11
444,"AQUINO, BALTAZAR",NaN,Batasang Bayan,1976.0,*Member of the Cabinet (Secretary of Public Hi...,20
445,"AQUINO, BALTAZAR",NaN,Interim Batasang Pambansa,1978.0,*Member of the Cabinet (Ministry of Public Hig...,21
537,"ASPIRAS, JOSE D.",NaN,Batasang Bayan,1976.0,*Member of the Cabinet (Secretary of Tourism).,24
1132,"CERILLES, VICENTE M.",NaN,Batasang Bayan,1976.0,*Regional Representative-at-Large.,48
1202,"CLAVE, JACOBO C.",NaN,Batasang Bayan,1976.0,*Member of the Cabinet (Presidential Executive...,51


In [119]:
roster.loc[roster["is_party_list"], "region_province"].value_counts().head(10)

region_province
Party-List    273
Name: count, dtype: int64

In [120]:
explained = no_prov["notes"].str.contains("Cabinet|at-Large", case=False, na=False)
print("explained by notes:", explained.sum(), "of", len(no_prov))
no_prov.loc[~explained, ["name", "district", "period_clean", "start_year", "notes", "page"]]

explained by notes: 48 of 64


,name,district,period_clean,start_year,notes,page
1404,"DATUMANONG, SIMEON A.",NaN,Batasang Bayan,1976.0,NaN,59
1479,"DE VEYRA, JAIME C.",Fourth,2nd Philippine Legislature,1910.0,NaN,63
1594,"DIEZ, CLEMENTINO V.",NaN,1st Commonwealth National Assembly,1935.0,*Unseated after losing his election protest. R...,67
1611,"DIMAPORO, IMELDA Q.",NaN,16th Congress,2013.0,NaN,68
1987,"FLORES, FLORENCIO JR. T.",Second,16th Congress,2013.0,NaN,83
1988,"FLORES, FLORENCIO JR. T.",Second,17th Congress,2016.0,NaN,83
2689,"LEVISTE, JOSE ANTONIO C.",NaN,Batasang Bayan,1976.0,NaN,112
2841,"LUKBAN, JUSTO R.",NaN,2nd Philippine Legislature,1910.0,"*Resigned on February 26, 1911 when Representa...",118
2930,"MAKALINTAL, QUERUBE C.",NaN,Interim Batasang Pambansa,1978.0,NaN,122
3110,"MENDOZA, ESTELITO P.",NaN,Interim Batasang Pambansa,1978.0,NaN,129


In [121]:
modern = roster[(roster["start_year"] >= 2001) & ~roster["is_party_list"] & ~roster["is_sectoral"]]
roster_prov = set(modern["region_province"].dropna().str.upper())

print("Roster places (2001+):", len(roster_prov))
print("Only in roster:", sorted(roster_prov - oh_prov))
print("Only in OpenHalalan:", sorted(oh_prov - roster_prov))

Roster places (2001+): 138
Only in roster: ['ANTIPOLO', 'ANTIPOLO CITY', 'BACOLOD CITY', 'BAGUIO CITY', 'BIÑAN CITY', 'CAGAYAN DE ORO CITY', 'CALOOCAN CITY', 'CEBU CITY', 'COMPOSTELA VALLEY', 'COMPOSTELLA VALLEY', 'DAVAO CITY', 'DINAGAT ISLAND', 'DINAGAT ISLAND*', 'ILIGAN CITY', 'ILIGAN CITY*', 'ILOILO CITY', 'LAPU-LAPU CITY', 'LAS PIÑAS', 'LAS PIÑAS CITY', 'LIPA CITY', 'MAGUINDANAO AND COTABATO CITY', 'MAGUINDANAO WITH COTABATO CITY', 'MAKATI', 'MAKATI CITY', 'MALABON / NAVOTAS', 'MALABON CITY', 'MALABON-NAVOTAS', 'MALABON-NAVOTAS CITY', 'MANDALUYONG CITY', 'MANILA', 'MARIKINA CITY', 'MUNTINLUPA CITY', 'NAVOTAS CITY', 'NORTH COTABATO', 'NORTH COTOBATO', 'NUEVA VISCAYA', 'PARAÑAQUE CITY', 'PART- LIST', 'PARTY LIST', 'PASAY CITY', 'PASIG', 'PASIG CITY', 'PATY-LIST', 'QUEZON CITY', 'SAN JOSE DEL MONTE CITY', 'SAN JUAN CITY', 'SHARIFF KABUNSUAN* WITH COTABATO CITY', 'SOUTH COTABATO AND GENERAL SANTOS CITY', 'SOUTH COTOBATO', 'TAGUIG CITY', 'TAGUIG CITY-PATEROS', 'TAGUIG CITY/PATEROS', 'TA

In [122]:
party_typos = modern["region_province"].str.upper().isin(["PART- LIST", "PARTY LIST", "PATY-LIST"])
print("party-list rows missed by the flag:", party_typos.sum())

party-list rows missed by the flag: 14


In [123]:
print("Latest start year:", roster["start_year"].max())
roster["period_clean"].value_counts().tail(10)

Latest start year: 2019.0


period_clean
Regular Batasang Pambasa                1
Third Philippine Legislature            1
9th Congresss                           1
Fourth Philippine Legislature           1
Ninth Philippine Legislature            1
Tenth Philippine Legislature            1
First National Assembly Commonwealth    1
12nd Congress                           1
Interim Batasang Pamabansa              1
1st National Assembly Commonwealth      1
Name: count, dtype: int64

### Summary: roster_legislators (staging)

**What it is:** The historical roster of the House of Representatives (and earlier legislative bodies), parsed from `roster_legislators.pdf` into a CSV (staging). 5,118 rows, 15 columns, one row per legislator per term. It runs from the 1st Philippine Legislature to terms starting in 2019 (18th Congress).

**Columns:** `name`, `surname`, `region_province`, `district`, `period_clean`, `congress_number`, `start_year`, `end_year`, `years_raw`, `years_corrected`, `years_suspect`, `notes`, `is_party_list`, `is_sectoral`, `page` (source PDF page).

**Strengths**
- **Decades of history**, which is valuable for kinship (task 7). Example: the Abad family held the Batanes lone district across generations (Jorge 1949–1953, Aurora 1966–1969, Florencio 1987–2004, Henedina 2004–2017).
- **`notes` (387 rows)** explain deaths, resignations, unseated members, and non-district seats.
- **Parser quality flags:** `years_corrected` (5 rows) and `years_suspect` (5 rows).
- **`page`** traces each row back to the PDF.

**Names**
- Format is "SURNAME, FIRST M.", with a **middle initial only** (OpenHalalan has full middle names).
- The suffix can sit **between the first name and the middle initial**, e.g. "COJUANGCO, EDUARDO JR. M." and "FLORES, FLORENCIO JR. T."
- **Compound surnames** such as "ABAD SANTOS" are separate families from "ABAD", so surnames must be used in full.

**Periods and congress numbers**
- `congress_number` is filled in only 3,217 of 5,118 rows. Most blanks are expected: older bodies without "Nth Congress" numbering (Philippine Legislatures, Commonwealth National Assemblies, Batasang Bayan, Interim and Regular Batasang Pambansa).
- **The parser missed some real Congresses** because of how they were written: words ("First Congress", "Second Congress", "Seventh Congress"), typos ("13th Congresss", "9th Congresss"), a wrong ordinal ("12nd Congress"), and lowercase ("12th congress"). "First Congress" is ambiguous (Commonwealth or Republic), so `start_year` is needed to resolve it.
- **`period_clean` has variants of the same body:** typos ("Philipine", "Pamabansa", "Pambasa"), words vs numerals ("Tenth" vs "10th"), and several spellings of the Commonwealth National Assembly.

**Party-list and sectoral**
- 273 rows are flagged `is_party_list`, with "Party-List" as a placeholder in `region_province` (not a real place).
- **14 more party-list rows were missed** by the flag because the placeholder was misspelled ("PARTY LIST", "PART- LIST", "PATY-LIST"). The true count is at least 287.
- 80 rows are flagged `is_sectoral`.

**Missing province (64 rows)**
- **48 are explained by notes:** Cabinet members and Regional Representatives-at-Large, who did not represent a province.
- **16 are unexplained:**
  - **3 modern rows** inside the HF/OpenHalalan years: DIMAPORO, IMELDA Q. (2013) and FLORES, FLORENCIO JR. T. (2013, 2016). These can likely be recovered from OpenHalalan by matching name and year.
  - **5 Marcos-era rows** (Datumanong, Leviste, Makalintal, Mendoza, Ponce Enrile), likely non-district members whose footnote was not captured.
  - **8 older rows (1910–1954)**, likely parsing gaps (e.g. De Veyra has a district but no province).
- `district` is missing in 670 rows (not yet investigated; partly party-list and sectoral).

**Place names (2001 onward, district representatives only)**
- 138 distinct place names, the messiest of all sources. Differences from OpenHalalan:
  - **Cities listed as provinces** (e.g. QUEZON CITY, MANILA, CEBU CITY), since House districts in large cities belong to the city.
  - **"CITY" variants** (MAKATI vs MAKATI CITY).
  - **Footnote asterisks** in names (ILIGAN CITY\*, DINAGAT ISLAND\*).
  - **Typos** (COMPOSTELLA VALLEY, NORTH COTOBATO, NUEVA VISCAYA, ZAMBOAGA SIBUGAY).
  - **Old or alternate names** (COMPOSTELA VALLEY, WESTERN SAMAR, NORTH COTABATO, DINAGAT ISLAND, TAWI-TAWI, SHARIFF KABUNSUAN).
  - **Shared districts** covering two places (TAGUIG-PATEROS, MALABON-NAVOTAS, SOUTH COTABATO AND GENERAL SANTOS CITY, MAGUINDANAO WITH COTABATO CITY), each written several ways.
- Names only in OpenHalalan are naming differences (DAVAO DE ORO, DINAGAT ISLANDS, SAMAR, TAWI TAWI, NCR districts) or 2025-only provinces, consistent with the roster ending in 2019.

## 6. Cleaning rules for task 6

### A. General
- Never edit files in `data/raw`. Cleaned outputs go to `data/staging`.
- Standardize column names to snake_case (OpenHalalan uses e.g. `Last Name`).
- Use **OpenHalalan's province names as the standard**. PSA already matches them, and HF needs only 6 mappings.
- Read folder paths from `.env`, with a local fallback, so the code works both locally and in Docker.
- Always match places on **province + town together** (e.g. ISABELA exists in both Basilan and Negros Occidental).

### B. Person names
1. **Surname particles:** standardize variants such as "De La Cruz" / "Dela Cruz" and "Vera Cruz" / "Veracruz". Merge only known variants with specific rules: "Cruz", "Cruzado", and "Cruzat" are different names.
2. **Abbreviations:** "Sta" → "Santa", "Sto" → "Santo".
3. **Suffixes:**
   - HF: move suffixes out of `last_name` into `name_suffix` (6 rows, e.g. "Lopez Jr" → "Lopez" + "Jr.").
   - Roster: handle the "SURNAME, FIRST JR. M." format, where the suffix sits between the first name and the middle initial.
   - Standardize to "Jr.", "Sr.", and Roman numerals. Flag implausible values ("XVI", "XIV") and review "I" (14 rows).
4. **Compound surnames** (e.g. "Abad Santos") are kept whole, never split.
5. **Hyphenated surnames** (e.g. "Escanlar-Cruz") are kept as is and flagged for the kinship decision (team).
6. **Roster name parsing:** split "SURNAME, FIRST M." into surname, first name, middle initial, and suffix.
7. **OpenHalalan missing names:** 7 missing last names and 82 missing first names. Flag them, since they cannot be matched by name.
8. **Middle name reliability:** keep `Middle Name Source`. Treat `self-prior`, `v8.5`, and rows with no source (10,194) as lower reliability.

### C. Provinces
1. **HF → standard:** "NCR, CITY OF MANILA, FIRST DISTRICT" → "NCR FIRST DISTRICT"; "NCR, SECOND/THIRD/FOURTH DISTRICT" → "NCR SECOND/THIRD/FOURTH DISTRICT"; "TAWI-TAWI" → "TAWI TAWI"; "COMPOSTELA VALLEY" → "DAVAO DE ORO".
2. **PSA → standard:** NCR district keys ("1ST DISTRICT", etc.; verify exact spellings) → the NCR district names. COTABATO CITY and ISABELA CITY are cities, not provinces: Cotabato is in MAGUINDANAO (MAGUINDANAO DEL NORTE from 2025), Isabela is in BASILAN.
3. **Roster → standard:**
   - Uppercase, and remove footnote asterisks (`*`).
   - Fix typos: COMPOSTELLA VALLEY, NORTH COTOBATO, SOUTH COTOBATO, NUEVA VISCAYA, ZAMBOAGA SIBUGAY.
   - Old or alternate names: COMPOSTELA VALLEY → DAVAO DE ORO, WESTERN SAMAR → SAMAR, DINAGAT ISLAND → DINAGAT ISLANDS, TAWI-TAWI → TAWI TAWI, NORTH COTABATO and SHARIFF KABUNSUAN → verify against OpenHalalan's names.
   - Cities listed as provinces (e.g. QUEZON CITY, MANILA, CEBU CITY): map each to its province or NCR district.
   - Shared districts (TAGUIG-PATEROS, MALABON-NAVOTAS, SOUTH COTABATO AND GENERAL SANTOS CITY, MAGUINDANAO WITH COTABATO CITY): standardize their spellings and apply a team-agreed rule.
   - Party-list placeholders ("Party-List", "PARTY LIST", "PART- LIST", "PATY-LIST"): set the province to blank and `is_party_list` to True (fixes 14 missed rows).
4. **Post-2016 changes:** the Maguindanao split (PSA 2023, OpenHalalan 2025) and Special Geographic Area (OpenHalalan 2025). Handle these according to the team decision on years.

### D. Towns
1. Apply `norm_town` to every town name: uppercase, remove parentheses, Ñ → N, remove hyphens, periods, and apostrophes, collapse spaces, remove "CITY OF" / " CITY".
2. Build a **manual town mapping table** (about 100 entries: 96 OpenHalalan names + 5 HF names), covering typos (MAGADALENA, SAM NARCISO), abbreviations (PRES CARLOS P GARCIA, D V CHIONGBIAN D M MARCOS), old-name variants (AMLAN AYUQUITAN), renamed towns (AMAI MANABILANG / BUMBARAN), respellings (BALIWAG), and combined entries (TAGUIG PATEROS).
3. Code suggests matches and a person approves them; never auto-apply loose matches (e.g. San Jose vs San Jose de Buan).
4. Store the mapping as a small table (`province`, `raw_name`, `standard_name`). Check `.gitignore` first so it is not blocked as a data file.

### E. Locality backfilling (HF memberships)
- **Target:** 22,022 Councilor, Mayor, and Vice Mayor rows with no locality (2013 and 2016).
- **Leave as blank:** the 5,618 provincial and district positions (expected, not errors).
- **Step 1, own history:** same person, same province, exactly one known town → fill (13,748 rows).
- **Step 2, ambiguous:** two or more known towns (80 rows) → nearest-year rule or OpenHalalan.
- **Step 3, OpenHalalan:** remaining rows (8,194) → match on name + province + year + position, after names and places are standardized.
- **Record how each value was filled** in a new column, e.g. `locality_source` = original / history / openhalalan / unfilled.
- **Validate step 1:** hide a complete year (e.g. 2010), refill it from history, and measure accuracy.
- Review the 96 rows whose known town is in a different province.
- Report how many rows remain unfilled.

### F. Dataset-specific checks
- **HF:** check the 39 House Representatives and 2 Provincial Board Members that have a locality. Check for possible merged people (one `id` in distant provinces in the same year).
- **OpenHalalan:** exclude national positions (Senator, President, Vice President) from town-level analysis. Investigate the 193 extra rows in 2016. `Sex` and `Title` are not yet profiled.
- **PSA:** add an `is_combined_area` flag for the 2023 Maguindanao row. Decide whether Isabela City and Cotabato City use their own figure or their province's. Confirm the meaning of the flags and whether figures refer to families or individuals.
- **Roster:**
  - Fix `congress_number` for word ordinals, typos ("Congresss", "12nd"), and lowercase. Use `start_year` to resolve "First Congress".
  - Build a `period_clean` standardization map.
  - Recover the 3 modern missing provinces (Dimaporo 2013, Flores 2013 and 2016) from OpenHalalan.
  - Review the 5 `years_suspect` rows.

### G. Open questions for the team
1. Analyze 2004–2016 only, or include OpenHalalan's 2001 and 2019–2025? (Poverty data exists only for 2018, 2021, and 2023.)
2. How should the Maguindanao split be handled?
3. How should hyphenated surnames count in kinship?
4. How should shared districts (e.g. Taguig-Pateros) be assigned?